# W03-B · Unseen word pairs, and scoring the model

**Your job.** A café app wants word suggestions in its order box. You have the bigram model from W03-A, trained on
12 sentences. Before anyone uses it, answer two questions:

1. What happens when a customer types something the model has **never seen**? (Right now: probability 0.)
2. How **good** is the model? Is it better than ignoring the previous word?

To answer the second one fairly, we test the model on **4 new sentences** it did not learn from, the way an exam
uses questions you haven't seen before.

| Task | What you do |
|---|---|
| 1 | Find the test pairs the model has never seen |
| 2 | Add-one smoothing: no more zeros |
| 3 | Score the test sentences again |
| 4 | Perplexity: one number for how surprised the model is |
| 5 | How much should we add? |

Every ✍️ task checks itself.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shripada/ads6703-nlp/blob/main/w03/W03-B-smoothing-and-perplexity.ipynb) · On Colab, run the next cell first. On your laptop, you can skip it.

In [ ]:
# Only for Google Colab: fetch the course folder (with the data) and move into this week's folder.
# On your laptop this cell does nothing.
import os, subprocess, sys
if "google.colab" in sys.modules:
    if os.path.exists("/content/ADS6703"):
        subprocess.run(["git", "-C", "/content/ADS6703", "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "https://github.com/shripada/ads6703-nlp.git", "/content/ADS6703"], check=True)
    os.chdir("/content/ADS6703/w03")

In [ ]:
# ▶ Setup: just run this cell. The training sentences and the counts are the same as in W03-A.
from collections import Counter

train = [
    "i like tea",
    "i like coffee",
    "you like tea",
    "i drink tea",
    "i like hot tea",
    "you drink coffee",
    "we like coffee",
    "we drink hot coffee",
    "i like cold coffee",
    "you like hot tea",
    "we like tea",
    "i drink coffee",
]
test = [                  # 4 new sentences: the model never counts these
    "you like coffee",
    "we drink tea",
    "i drink cold tea",
    "you like cold coffee",
]

def add_marks(sentence):
    return ["<s>"] + sentence.split() + ["</s>"]

pair_counts = Counter()   # (word, next word) -> how often, in the training sentences
word_counts = Counter()   # word -> how often it appears as a next word (so </s> counts, <s> doesn't)
for s in train:
    marked = add_marks(s)
    pair_counts.update(zip(marked, marked[1:]))
    word_counts.update(marked[1:])

def next_counts(word):
    # W03-A, Exercise 3: each word seen right after `word` -> how many times
    result = {}
    for (w1, w2), n in pair_counts.items():
        if w1 == word:
            result[w2] = n
    return result

def p_plain(w1, w2):
    # W03-A, Exercise 4: the bigram probability with no smoothing
    after = next_counts(w1)
    if len(after) == 0:
        return 0
    return after.get(w2, 0) / sum(after.values())

V = len(word_counts)      # how many different words can come next
print("V =", V, sorted(word_counts))

test_pairs = []           # every step the model has to guess in the test sentences
for s in test:
    marked = add_marks(s)
    test_pairs = test_pairs + list(zip(marked, marked[1:]))
print(len(test_pairs), "test pairs")

## Task 1 · Which test pairs has the model never seen?

Every test pair that never appeared in training gets probability 0.

### ✍️ Exercise 1: the unseen pairs

Make `unseen`: the list of pairs in `test_pairs` whose count in `pair_counts` is 0. (Looking up a missing pair in a
`Counter` gives 0, no error.)

In [ ]:
unseen = []
for pair in test_pairs:
    ...   # ✍️ add the pair to `unseen` if it was never seen in training
print(unseen)
assert unseen == [("drink", "cold"), ("cold", "tea")], "not quite"
print("✅")

Both unseen pairs are in *"i drink cold tea"*. So without smoothing, this ordinary sentence gets probability 0:

In [ ]:
def sentence_prob(sentence, prob):
    # Multiply the probability of every step. `prob` is the function that gives one step's probability.
    words = add_marks(sentence)
    p = 1
    for w1, w2 in zip(words, words[1:]):
        p = p * prob(w1, w2)
    return p

for s in test:
    print(f"{sentence_prob(s, p_plain):.4f}   {s}")

## Task 2 · Add-one smoothing: no more zeros

The fix is simple: **pretend you saw every possible pair one more time than you really did.** A pair seen 0 times now
counts as 1, a pair seen 2 times counts as 3. This is called **add-one smoothing** (or *Laplace smoothing*).

After a word, there are `V` possible next words, and each one gets 1 extra count. So the bottom of the fraction grows by `V`:

$$P_{\text{add-1}}(\text{next} \mid \text{word}) = \frac{\text{count(word next)} + 1}{\text{count(word)} + V}$$

Example: after *drink* we saw *tea* 1, *coffee* 2, *hot* 1: 4 in all. With V = 10:
P(cold \| drink) = (0 + 1) ÷ (4 + 10) = 1/14, and P(coffee \| drink) = (2 + 1) ÷ (4 + 10) = 3/14.

### ✍️ Exercise 2: `p_add_one(w1, w2)`

Hint: `after = next_counts(w1)`; the count of the pair is `after.get(w2, 0)`, and count(word) is `sum(after.values())`.

In [ ]:
def p_add_one(w1, w2):
    after = next_counts(w1)
    return ...   # ✍️ (count of the pair + 1) / (count of w1 + V)
print("P(cold | drink):  no smoothing", p_plain("drink", "cold"), "  add-one", p_add_one("drink", "cold"))
print("P(coffee | drink): no smoothing", p_plain("drink", "coffee"), "  add-one", p_add_one("drink", "coffee"))
assert abs(p_add_one("drink", "cold") - 1 / 14) < 1e-12, "not quite: (0 + 1) / (4 + 10)"

# The probabilities after one word must still add up to 1
total = 0
for w in word_counts:
    total = total + p_add_one("drink", w)
assert abs(total - 1) < 1e-9, "the probabilities after 'drink' should add up to 1"
print("✅")

> 📖 **Reading the output.** *drink cold* is no longer impossible: 1/14. But look at *drink coffee*: it fell from
> 0.5 to about 0.21. To give something to the pairs we never saw, smoothing **takes probability away** from the pairs
> we did see. Keep that in mind for Task 5.

## Task 3 · Score the test sentences again

### ✍️ Exercise 3

Use `sentence_prob` with your `p_add_one` to fill the list `smoothed`: one probability per test sentence, in order.

In [ ]:
smoothed = []
for s in test:
    ...   # ✍️ append the add-one probability of s (hint: sentence_prob(s, p_add_one))
for p, s in zip(smoothed, test):
    print(f"{p:.5f}   {s}")
assert len(smoothed) == 4 and min(smoothed) > 0, "not quite: every test sentence should now be above 0"
print("✅")

> 📖 **Reading the output.** All four sentences are now above 0. *"i drink cold tea"* is still the least likely,
> which is fair: it uses two pairs the model never saw.

## Task 4 · Perplexity: how surprised is the model?

To score a model, let it read the test sentences one word at a time. Before each word it gives the real next word a
probability. A good model gives the words that really come next **high** probabilities.

**Perplexity** turns that into one number: *on average, how many words was the model choosing between?*

- If the model gave every real next word probability ½, it was as unsure as a coin toss: perplexity **2**.
- If it gave each real word ¼, it was choosing between 4 words: perplexity **4**.
- **Lower is better.** A perfect model, always sure and always right, has perplexity 1.

When the probabilities differ, multiply them all, and take the *N*-th root (*N* = the number of probabilities):

$$\text{perplexity} = \left(\frac{1}{p_1 \times p_2 \times \dots \times p_N}\right)^{1/N}$$

By hand, with *N* = 2: ½ × ⅛ = 1/16; 1 ÷ (1/16) = 16; the square root of 16 is **4**. In Python, the *N*-th root of
`x` is `x ** (1 / N)`.

### ✍️ Exercise 4: `perplexity(prob)`

Multiply `prob(w1, w2)` over every pair in `test_pairs`, then apply the formula.

In [ ]:
def perplexity(prob):
    product = 1
    for w1, w2 in test_pairs:
        ...   # ✍️ multiply product by this pair's probability
    N = len(test_pairs)
    return ...   # ✍️ (1 / product) to the power 1 / N
def always_a_quarter(w1, w2):
    return 0.25                  # a silly model that gives every word probability 1/4
assert abs(perplexity(always_a_quarter) - 4) < 1e-9, "a model that always says 1/4 should have perplexity 4"
print("✅")

Now score two real models. The **unigram** model ignores the previous word (with add-one, so it never gives 0);
the **bigram** model with add-one uses your `p_add_one`.

In [ ]:
total_words = sum(word_counts.values())

def p_unigram(w1, w2):
    # Ignores w1 completely: just how common w2 is (with add-one)
    return (word_counts[w2] + 1) / (total_words + V)

print(f"unigram (no context):  perplexity {perplexity(p_unigram):.1f}")
print(f"bigram with add-one:   perplexity {perplexity(p_add_one):.1f}")
# What about the bigram with no smoothing? Try it: perplexity(p_plain). Why does it fail?

> 📖 **Reading the output.** The unigram model is choosing between about 10 words at every step: almost every word
> it knows, so it has learnt very little. The bigram model is choosing between about 5: looking at the previous word
> halves its confusion.
>
> And `perplexity(p_plain)` fails with a *division by zero*: one zero probability makes the product 0, and 1 ÷ 0
> is impossible. That is the zero problem again, now in a single number.

## Task 5 · How much should we add?

In Task 2 we saw smoothing take probability away from pairs we did see. Maybe adding a whole 1 is too much. Try
adding a smaller amount `k` instead (**add-k smoothing**): add `k` on top, and `k × V` at the bottom.

$$P_{\text{add-k}}(\text{next} \mid \text{word}) = \frac{\text{count(word next)} + k}{\text{count(word)} + k \times V}$$

### ✍️ Exercise 5: `p_add_k`, and the best `k`

Write `p_add_k(w1, w2)` using the variable `k`. The loop below tries five values of `k` and prints the perplexity of each.
Then set `best_k` to the `k` with the **lowest** perplexity.

In [ ]:
def p_add_k(w1, w2):
    after = next_counts(w1)
    return ...   # ✍️ like p_add_one, with k instead of 1 on top, and k * V at the bottom

for k in [1, 0.5, 0.1, 0.01, 0.001]:      # p_add_k uses whatever k is set to here
    print(f"k = {k:<6} perplexity {perplexity(p_add_k):.2f}")

best_k = ...   # ✍️ read it off the table above
assert best_k == 0.1, "not quite: which k has the smallest perplexity?"
print(f"✅ best k = {best_k}")

> 📖 **Reading the output.** Adding less than 1 helps, down to about 3.4 with `k` = 0.1. But add too little
> (0.001) and it gets worse again, because the unseen pairs *drink cold* and *cold tea* become almost impossible.
> There is a best amount in between, and the only way to find it is to **try values on test sentences**.
>
> Why not choose `k` on the training sentences? There, every pair has been seen, so the best `k` would always be 0:
> no smoothing at all. Only new sentences show how well the model copes with what it hasn't read.

## Summary

| Model, tested on 4 new sentences | Perplexity |
|---|---|
| unigram: ignore the previous word | about 9.8 |
| bigram, no smoothing | can't be computed: a zero |
| bigram, add-one | about 4.9 |
| bigram, add-0.1 | about 3.4 |